In [1]:
import numpy as np
import cv2
from PIL import Image
import pickle
import os

output_dir = "comic/phase1_output"
os.makedirs(output_dir, exist_ok=True)

# Load and resize image
img = np.array(Image.open("comic/comic_1.webp").convert("RGB"))
scale = 800 / img.shape[1]
new_height = int(img.shape[0] * scale)
img = cv2.resize(img, (800, new_height))
height, width, _ = img.shape

print(f"Image shape: {height}x{width}")

# Save original image
original_path = os.path.join(output_dir, "00_original_image.jpg")
cv2.imwrite(original_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
print(f"Original image saved: {original_path}")

Image shape: 393x800
Original image saved: comic/phase1_output\00_original_image.jpg


In [2]:
# ===== MANUAL PARAMETERS (adjust here) =====
ctol = 30      # Color tolerance
msz = 2        # Minimum size
ptol = 3       # (Not used in Phase 1, kept for reference)

print(f"Parameters: ctol={ctol}, msz={msz}")

Parameters: ctol=30, msz=2


In [3]:
region_map = np.full((height, width), -1, dtype=int)
region_colors = []
region_id = 0

for y in range(height):
    for x in range(width):
        if region_map[y, x] != -1:
            continue

        pixel_color = img[y, x].astype(float)
        region_map[y, x] = region_id
        region_colors.append(pixel_color.copy())

        queue = [(y, x)]
        region_pixels = [(y, x)]

        while queue:
            cy, cx = queue.pop(0)
            avg_color = region_colors[region_id]

            for ny, nx in [(cy-1, cx), (cy+1, cx), (cy, cx-1), (cy, cx+1)]:
                if 0 <= ny < height and 0 <= nx < width:
                    if region_map[ny, nx] == -1:
                        neighbor_color = img[ny, nx].astype(float)
                        diff = np.sqrt(np.sum((neighbor_color - avg_color) ** 2))
                        if diff < ctol:
                            region_map[ny, nx] = region_id
                            region_pixels.append((ny, nx))
                            queue.append((ny, nx))
                            n = len(region_pixels)
                            region_colors[region_id] = avg_color + (neighbor_color - avg_color) / n

        region_id += 1

print(f"Total regions after BFS: {region_id}")

Total regions after BFS: 38238


In [4]:
region_neighbors = [[] for _ in range(region_id)]
rsizes = np.zeros(region_id)
rcolors = region_colors.copy()

for rid in range(region_id):
    rsizes[rid] = np.sum(region_map == rid)

for y in range(height):
    for x in range(width):
        rid = region_map[y, x]
        for ny, nx in [(y-1, x), (y+1, x), (y, x-1), (y, x+1)]:
            if 0 <= ny < height and 0 <= nx < width:
                nid = region_map[ny, nx]
                if nid != rid and nid not in region_neighbors[rid]:
                    region_neighbors[rid].append(nid)

def rgb_to_y(color):
    r, g, b = color[0], color[1], color[2]
    return 0.299 * r + 0.587 * g + 0.114 * b

rmap = region_map.copy()

for rid in range(region_id):
    if rsizes[rid] > 0 and rsizes[rid] < msz:
        y_value = rgb_to_y(rcolors[rid])
        best_neighbor = -1
        best_diff = float('inf')
        best_size = 0

        if y_value < 100:
            for nid in region_neighbors[rid]:
                if rsizes[nid] > 0:
                    diff = np.sqrt(np.sum((np.array(rcolors[rid]) - np.array(rcolors[nid])) ** 2))
                    if diff < best_diff:
                        best_diff = diff
                        best_neighbor = nid
        else:
            for nid in region_neighbors[rid]:
                if rsizes[nid] >= msz:
                    diff = np.sqrt(np.sum((np.array(rcolors[rid]) - np.array(rcolors[nid])) ** 2))
                    if rsizes[nid] > best_size or (rsizes[nid] == best_size and diff < best_diff):
                        best_diff = diff
                        best_neighbor = nid
                        best_size = rsizes[nid]

        if best_neighbor != -1:
            rmap[rmap == rid] = best_neighbor
            rsizes[best_neighbor] += rsizes[rid]
            rsizes[rid] = 0

remaining = len(np.unique(rmap[rmap >= 0]))
print(f"Regions after merging (msz={msz}): {remaining}")

Regions after merging (msz=2): 13658


In [5]:
phase1_results = {
    'img': img,                    
    'height': height,
    'width': width,
    'region_map': rmap,            
    'region_id': region_id,
    'rsizes': rsizes,
    'rcolors': rcolors,
    'ctol': ctol,
    'msz': msz
}

with open('phase1_results.pkl', 'wb') as f:
    pickle.dump(phase1_results, f)

print("Phase 1 results saved to phase1_results.pkl")
print(f"Key outputs:")
print(f"  - img: original image")
print(f"  - region_map: region segmentation")
print(f"  - Regions: {remaining}")

Phase 1 results saved to phase1_results.pkl
Key outputs:
  - img: original image
  - region_map: region segmentation
  - Regions: 13658


In [6]:
# Visualize region segmentation with different colors
np.random.seed(42)
color_map = np.random.randint(0, 256, size=(region_id, 3), dtype=np.uint8)
color_map[0] = [0, 0, 0]  # Region 0 as black

vis_segmentation = np.zeros((height, width, 3), dtype=np.uint8)

for y in range(height):
    for x in range(width):
        rid = rmap[y, x]
        if rid >= 0:
            vis_segmentation[y, x] = color_map[rid]

# Save segmentation visualization
seg_path = os.path.join(output_dir, "01_segmentation_map.jpg")
cv2.imwrite(seg_path, vis_segmentation)
print(f"Segmentation map saved: {seg_path}")

# Statistics
print(f"\nSegmentation Statistics:")
print(f"  Total regions: {remaining}")
print(f"  Average region size: {height*width / remaining:.1f} pixels")
print(f"  Largest region: {np.max(rsizes):.0f} pixels")
print(f"  Smallest region: {np.min(rsizes[rsizes>0]):.0f} pixels")

Segmentation map saved: comic/phase1_output\01_segmentation_map.jpg

Segmentation Statistics:
  Total regions: 13658
  Average region size: 23.0 pixels
  Largest region: 85886 pixels
  Smallest region: 1 pixels
